# Clases Internas (Nested/Inner Classes) en Python

En Python, **una clase interna (o anidada) es simplemente una clase definida dentro del cuerpo de otra clase**.

Son una herramienta excelente para mejorar la **encapsulación**, el **agrupamiento lógico** y la **legibilidad** del código.
___
## 1. Sintaxis Básica y Creación de Instancias

Para definir una **clase interna**, colocamos su definición dentro de la **clase contenedora (externa)**.

### Regla fundamental:
La clase interna es un **atributo** de la clase externa.

In [1]:
class Externa:
    def __init__(self, nombre_externo):
        self.nombre_externo = nombre_externo

    class Interna:
        def __init__(self, nombre_interno):
            self.nombre_interno = nombre_interno
            
        def mostrar_detalles(self):
            return f"Soy la clase interna. Mi nombre es: {self.nombre_interno}"

___
### Formas de instanciación de la clase interna
* Desde dentro de la externa
* Desde fuera de la externa

___
#### Desde dentro de la externa
Lo hacemos en el **__init__()** de la externa.

Para que se cree el objeto hay también que instanciár la externa.

In [1]:
class Externa:
    def __init__(self, nombre_externo):
        self.nombre_externo = nombre_externo
        # Instanciación de la clase interna dentro de la externa
        self.interna_por_defecto = self.Interna("Instancia automática")

    class Interna:
        def __init__(self, nombre_interno):
            self.nombre_interno = nombre_interno
            
        def mostrar_detalles(self):
            return f"Soy la clase interna. Mi nombre es: {self.nombre_interno}"

# Instanciación de la clase externa
objeto_externo = Externa("Principal")
print(objeto_externo.interna_por_defecto.mostrar_detalles())

Soy la clase interna. Mi nombre es: Instancia automática


___
#### Desde fuera de la externa
Lo hacemos desde fuera de cualquier clase.

La sintaxis debe incluir el nombre de la externa: `ClaseExterna.ClaseInterna()`

In [2]:
class Externa:
    def __init__(self, nombre_externo):
        self.nombre_externo = nombre_externo

    class Interna:
        def __init__(self, nombre_interno):
            self.nombre_interno = nombre_interno
            
        def mostrar_detalles(self):
            return f"Soy la clase interna. Mi nombre es: {self.nombre_interno}"

# Instanciación de la interna directamente desde fuera
# Sintaxixs ClaseExterna.ClaseInterna()
objeto_interno_directo = Externa.Interna("Instancia Manual")
print( objeto_interno_directo.mostrar_detalles())

Soy la clase interna. Mi nombre es: Instancia Manual


___
## El problema del acceso a los atributos del padre

A diferencia de otros lenguajes, **la clase interna en Python NO tiene acceso automático a los atributos de la instancia de la clase externa**. 

En Python, el anidamiento de clases es puramente estructural y cosmético (organizacional).

___
Fallará si intentamos acceder directamente a un atributo de la externa independientemente de cómo instanciemos la interna

#### Falla tanto la instanciar la interna desde la externa

In [4]:
# Instanciando la interna desde la externa
# FALLARÁ SI NO SE COMENTA LA ÚLTIMA LÍNEA

class ExternaIncorrecta:
    def __init__(self, valor):
        self.valor_externo = valor
        self.interna_por_defecto = self.Interna() # Instanciamos la interna desde la externa

    class Interna:
        def ver_externa(self):
            # Esto fallará porque 'self' aquí se refiere a Interna, no a ExternaIncorrecta
            return self.valor_externo 
            #pass

ei = ExternaIncorrecta(2)
# ei.interna_por_defecto.ver_externa() # ERROR: La interna no reconoce el atributo valor_externo porque ese self se refiere a la Interna

#### Falla también al instanciar la interna desde fuera

In [6]:
# Instanciando la interna desde fuera
# FALLARÁ SI NO SE COMENTA LA ÚLTIMA LÍNEA

class ExternaIncorrecta:
    def __init__(self, valor):
        self.valor_externo = valor

    class Interna:
        def ver_externa(self):
            # Esto fallará porque 'self' aquí se refiere a Interna, no a ExternaIncorrecta
            return self.valor_externo 
            #pass

ei = ExternaIncorrecta.Interna()
# ei.ver_externa() # ERROR: La interna no reconoce el atributo valor_externo porque ese self se refiere a la Interna

___
### Solución Correcta:
`Pasar la instancia del padre como argumento al construir la interna.`

In [7]:
class Humano:
    def __init__(self, nombre):
        self.nombre = nombre
        # Pasamos 'self' (la instancia de Humano) al constructor de la clase interna Cabeza
        self.cerebro = self.Cabeza(self) 

    class Cabeza:
        def __init__(self, humano_referencia):
            self.humano_referencia = humano_referencia  # Guardamos la referencia del objeto contenedor

        def pensar(self):
            return f"{self.humano_referencia.nombre} está pensando intensamente."


# Ejecución
persona = Humano("Carlos")
print(persona.cerebro.pensar())

Carlos está pensando intensamente.


___
## Uso típico: Estructuras de Datos
Las clases internas son importantes cuando una clase secundaria **no tiene sentido que exista por sí misma** fuera del contexto de la clase principal. 

Imaginemos un sistema de gestión de pedidos. Un pedido (`Pedido`) tiene múltiples líneas de detalle (`Item`), y estas líneas solo existen si el pedido existe.

In [9]:
class Pedido:
    def __init__(self, id_pedido, cliente):
        self.id_pedido = id_pedido
        self.cliente = cliente
        self.items = []  # Lista que guardará instancias de la clase interna

    # Método para agregar ítems usando la clase interna
    def agregar_producto(self, producto, cantidad, precio_unitario):
        # Instanciamos la clase interna dentro del método de la externa
        nuevo_item = self.Item(producto, cantidad, precio_unitario)
        self.items.append(nuevo_item)

    def calcular_total_pedido(self):
        return sum([item.calcular_subtotal() for item in self.items])

    def mostrar_factura(self):
        print(f"Factura ID: {self.id_pedido} | Cliente: {self.cliente}")
        print("-" * 40)
        for item in self.items:
            print(f"- {item.producto} x{item.cantidad}: {item.calcular_subtotal():.2f}€")
        print("-" * 40)
        print(f"TOTAL: {self.calcular_total_pedido():.2f}€\n")

    # --- CLASE INTERNA ---
    class Item:
        def __init__(self, producto, cantidad, precio_unitario):
            self.producto = producto
            self.cantidad = cantidad
            self.precio_unitario = precio_unitario

        def calcular_subtotal(self):
            return self.cantidad * self.precio_unitario

# --- Pruebas del mundo real ---
mi_pedido = Pedido(id_pedido=1024, cliente="Ana Martínez")
mi_pedido.agregar_producto("Portátil Asus", 1, 899.99)
mi_pedido.agregar_producto("Ratón Logitech", 2, 5.50)
mi_pedido.agregar_producto("Alfombrilla ratón", 4, 3.00)

mi_pedido.mostrar_factura()


Factura ID: 1024 | Cliente: Ana Martínez
----------------------------------------
- Portátil Asus x1: 899.99€
- Ratón Logitech x2: 11.00€
- Alfombrilla ratón x4: 12.00€
----------------------------------------
TOTAL: 922.99€



---

## Herencia en Clases Internas

Las clases internas también pueden heredar de otras clases (ya sean internas o externas). Esto añade una capa extra de potencia para arquitecturas de software orientadas a objetos complejas (por ejemplo, en frameworks de desarrollo web como Django, donde se usa mucho `class Meta`).

In [ ]:
class Computadora:
    class Componente:
        def __init__(self, marca):
            self.marca = marca
        def obtener_tipo(self):
            return "Componente Genérico"

    # Clase interna que hereda de otra clase interna
    class Procesador(Componente):
        def __init__(self, marca, nucleos):
            super().__init__(marca)
            self.nucleos = nucleos
            
        def obtener_tipo(self):
            return f"Procesador {self.marca} de {self.nucleos} núcleos"

    # Clase interna que hereda de otra clase interna
    class DiscoDuro(Componente):
        def __init__(self, marca, capacidad_gb):
            super().__init__(marca)
            self.capacidad_gb = capacidad_gb
            
        def obtener_tipo(self):
            return f"Disco Duro {self.marca} de {self.capacidad_gb}GB"

# Instanciación
cpu = Computadora.Procesador("AMD Ryzen", 8)
disco = Computadora.DiscoDuro("Kingston", 1000)

print(cpu.obtener_tipo())
print(disco.obtener_tipo())

---

## Atributos `__name__` y `__qualname__`

* `__qualname__`: (nombre cualificado) es una propiedad especial disponible en clases, funciones y métodos que devuelve la ruta completa y jerárquica del objeto desde el nivel superior del módulo. Devuelve el nombre acompañado de su contenedor (clase principal, método o módulo) separado por puntos.

* `__name__`: Devuelve únicamente el nombre simple o directo del objeto. Si tienes dos clases o funciones con el mismo nombre en diferentes contextos, __name__ se verá igual para ambas.

print("Nombre de la clase Externa:", Pedido.__name__)
print("Nombre calificado de la clase Externa:", Pedido.__qualname__)
print("=" * 5)
print("Nombre de la clase Interna:", Pedido.Item.__name__)
print("Nombre calificado de la clase Interna:", Pedido.Item.__qualname__)

# Comprobando tipos de las instancias
instancia_item = mi_pedido.items[0]
print("Tipo de la instancia interna:", type(instancia_item))


___
## Conclusiones: ¿Cuándo usarlas y cuándo evitarlas?

### Pros:
1. **Encapsulamiento Limpio:** Si una clase solo sirve para ayudar a otra, esconderla dentro evita "contaminar" el espacio de nombres global del módulo.
2. **Organización Lógica:** Hace evidente que dos conceptos están íntimamente relacionados (ej. `Pedido` e `Item`).

### Contras:
1. **Legibilidad:** Si las clases internas se vuelven muy grandes, el archivo de código se vuelve difícil de leer (se genera mucha indentación).
2. **Filosofía Python:** El Zen de Python dice *"Flat is better than nested"* (Plano es mejor que anidado). En Python, muchas veces es preferible simplemente poner ambas clases en el mismo archivo `.py` (módulo) una al lado de la otra, en lugar de meter una dentro de otra, a menos que la cohesión sea absoluta.
